# Введение в MapReduce модель на Python


In [11]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [12]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [13]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [14]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [15]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [16]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [17]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [18]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [19]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [20]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [21]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [22]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [23]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [24]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [25]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.185568040959623)),
 (1, np.float64(2.185568040959623)),
 (2, np.float64(2.185568040959623)),
 (3, np.float64(2.185568040959623)),
 (4, np.float64(2.185568040959623))]

## Inverted index

In [26]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('it', ['0', '1', '2']),
 ('is', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [27]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [28]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [29]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('a', 2), ('banana', 2), ('is', 18)]),
 (1, [('it', 18), ('what', 10)])]

## TeraSort

In [30]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.08319498833243877)),
   (None, np.float64(0.09571251661238711)),
   (None, np.float64(0.12062866599032374)),
   (None, np.float64(0.19422296057877086)),
   (None, np.float64(0.243666374536874)),
   (None, np.float64(0.2504553653965067)),
   (None, np.float64(0.3041207890271841)),
   (None, np.float64(0.3172854818203209)),
   (None, np.float64(0.3427638337743084)),
   (None, np.float64(0.4148262119536318)),
   (None, np.float64(0.4170222110247016)),
   (None, np.float64(0.48303426426270435))]),
 (1,
  [(None, np.float64(0.5104223374780111)),
   (None, np.float64(0.5194851192598093)),
   (None, np.float64(0.5450680064664649)),
   (None, np.float64(0.5724569574914731)),
   (None, np.float64(0.5859365525622129)),
   (None, np.float64(0.6030601284109274)),
   (None, np.float64(0.6128945257629677)),
   (None, np.float64(0.6249035020955999)),
   (None, np.float64(0.6693137829622723)),
   (None, np.float64(0.6746890509878248)),
   (None, np.float64(0.681300765792796

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [31]:
from typing import NamedTuple, Iterator

def MAP(_, row: NamedTuple):
    yield ("max", row.number)

def REDUCE(key: str, numbers: Iterator[int]):
    max_val = None
    for n in numbers:
        if max_val is None or n > max_val:
            max_val = n
    yield (key, max_val)

class NUM(NamedTuple):
    id: int
    number: int

input_collection = [
    NUM(id=0, number=55),
    NUM(id=1, number=100),
    NUM(id=2, number=25),
    NUM(id=3, number=101)
]

def RECORDREADER():
    return [(u.id, u) for u in input_collection]

def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()

map_output = list(flatten(map(lambda x: MAP(*x), RECORDREADER())))
shuffle_output = list(groupbykey(map_output))
reduce_output = list(flatten(REDUCE(k, iter(vs)) for k, vs in shuffle_output))

print("MAP output:", map_output)
print("SHUFFLE output:", shuffle_output)
print("RESULT:", reduce_output)

MAP output: [('max', 55), ('max', 100), ('max', 25), ('max', 101)]
SHUFFLE output: [('max', [55, 100, 25, 101])]
RESULT: [('max', 101)]


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [32]:
from typing import NamedTuple, Iterator

def MAP(_, row: NamedTuple):
    yield ("avg", (row.number, 1))

def REDUCE(key: str, pairs: Iterator[tuple[int, int]]):
    total_sum = 0
    total_cnt = 0
    for x, c in pairs:
        total_sum += x
        total_cnt += c

    mean = total_sum / total_cnt if total_cnt != 0 else None
    yield (key, mean)

class NUM(NamedTuple):
    id: int
    number: int

input_collection = [
    NUM(id=0, number=55),
    NUM(id=1, number=25),
    NUM(id=2, number=25),
    NUM(id=3, number=33),
]

def RECORDREADER():
    return [(u.id, u) for u in input_collection]

def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()

map_output = list(flatten(map(lambda x: MAP(*x), RECORDREADER())))
shuffle_output = list(groupbykey(map_output))
reduce_output = list(flatten(REDUCE(k, iter(vs)) for k, vs in shuffle_output))

print("MAP output:", map_output)
print("SHUFFLE output:", shuffle_output)
print("RESULT:", reduce_output)

MAP output: [('avg', (55, 1)), ('avg', (25, 1)), ('avg', (25, 1)), ('avg', (33, 1))]
SHUFFLE output: [('avg', [(55, 1), (25, 1), (25, 1), (33, 1)])]
RESULT: [('avg', 34.5)]


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [33]:
import itertools

def groupByKey_sort(pairs):
    pairs_sorted = sorted(pairs, key=lambda pair: pair[0])

    for key, group_iter in itertools.groupby(pairs_sorted, key=lambda pair: pair[0]):
        values = []
        for _, value in group_iter:
            values.append(value)
        yield (key, values)

pairs = [('b', 1), ('a', 2), ('b', 3), ('a', 4), ('c', 5), ('b', 6)]
print(list(groupByKey_sort(pairs)))

pairs = [('a', 10), ('a', 20), ('b', 30)]
print(list(groupByKey_sort(pairs)))

pairs = [('max', 5), ('max', -2), ('max', 10)]
print(list(groupByKey_sort(pairs)))

pairs = [('x', 9), ('a', 1), ('m', 5)]
print(list(groupByKey_sort(pairs)))


[('a', [2, 4]), ('b', [1, 3, 6]), ('c', [5])]
[('a', [10, 20]), ('b', [30])]
[('max', [5, -2, 10])]
[('a', [1]), ('m', [5]), ('x', [9])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [34]:
from typing import NamedTuple, Iterator

class NUM(NamedTuple):
    id: int
    number: int

input_collection = [
    NUM(0, 55),
    NUM(1, 25),
    NUM(2, 25),
    NUM(3, 33),
    NUM(4, 55),
    NUM(5, 33),
]

def RECORDREADER():
    return [(u.id, u) for u in input_collection]

def MAP(_, row: NUM):
    yield (row.number, 1)

def REDUCE(key: int, values: Iterator[int]):
    yield key

def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()


map_output = list(flatten(map(lambda x: MAP(*x), RECORDREADER())))
shuffle_output = list(groupbykey(map_output))
reduce_output = list(flatten(REDUCE(k, iter(vs)) for k, vs in shuffle_output))

print("MAP output:", map_output)
print("SHUFFLE output:", shuffle_output)
print("RESULT:", reduce_output)

MAP output: [(55, 1), (25, 1), (25, 1), (33, 1), (55, 1), (33, 1)]
SHUFFLE output: [(55, [1, 1]), (25, [1, 1]), (33, [1, 1])]
RESULT: [55, 25, 33]


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [35]:
from typing import NamedTuple, Iterator

class Person(NamedTuple):
    id: int
    name: str
    age: int

input_collection = [
    Person(0, "Ann", 17),
    Person(1, "Bob", 22),
    Person(2, "Eva", 19),
]

def RECORDREADER():
    return [(p.id, p) for p in input_collection]

def C(t: Person) -> bool:
    return t.age >= 18

def MAP(_, t: Person):
    if C(t):
        yield (t, t)

def REDUCE(key: Person, values: Iterator[Person]):
    for v in values:
        yield v

def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()

map_output = list(flatten(map(lambda x: MAP(*x), RECORDREADER())))
shuffle_output = list(groupbykey(map_output))
reduce_output = list(flatten(REDUCE(k, iter(vs)) for k, vs in shuffle_output))

print("MAP output:", map_output)
print("SHUFFLE output:", shuffle_output)
print("RESULT:", reduce_output)

MAP output: [(Person(id=1, name='Bob', age=22), Person(id=1, name='Bob', age=22)), (Person(id=2, name='Eva', age=19), Person(id=2, name='Eva', age=19))]
SHUFFLE output: [(Person(id=1, name='Bob', age=22), [Person(id=1, name='Bob', age=22)]), (Person(id=2, name='Eva', age=19), [Person(id=2, name='Eva', age=19)])]
RESULT: [Person(id=1, name='Bob', age=22), Person(id=2, name='Eva', age=19)]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [36]:
from typing import Iterator

R = [
    ("Ivan", 25, "Samara"),
    ("Olga", 31, "Moscow"),
    ("Petr", 40, "Samara"),
    ("Anna", 19, "Kazan"),
    ("Dmitry", 31, "Samara"),
    ("Dmitry", 45, "Samara"),
]

S = (0, 2)

def RECORDREADER():
    for t in R:
        yield (t, t)

def MAP(_, t):
    t_prime = tuple(t[i] for i in S)
    yield (t_prime, t_prime)

def REDUCE(key, values: Iterator[tuple]):
    yield (key, key)

def flatten(nested):
    for it in nested:
        for x in it:
            yield x

def groupbykey(pairs):
    d = {}
    for k, v in pairs:
        d.setdefault(k, []).append(v)
    return d.items()

map_output = list(flatten(MAP(*x) for x in RECORDREADER()))
shuffle_output = list(groupbykey(map_output))
reduce_output = list(flatten(REDUCE(k, iter(vs)) for k, vs in shuffle_output))

print("S (indices):", S)
print("MAP output:", map_output)
print("SHUFFLE output:", shuffle_output)
print("RESULT:", reduce_output)

projected = [k for (k, _) in reduce_output]
print("PROJECTED:", projected)

S (indices): (0, 2)
MAP output: [(('Ivan', 'Samara'), ('Ivan', 'Samara')), (('Olga', 'Moscow'), ('Olga', 'Moscow')), (('Petr', 'Samara'), ('Petr', 'Samara')), (('Anna', 'Kazan'), ('Anna', 'Kazan')), (('Dmitry', 'Samara'), ('Dmitry', 'Samara')), (('Dmitry', 'Samara'), ('Dmitry', 'Samara'))]
SHUFFLE output: [(('Ivan', 'Samara'), [('Ivan', 'Samara')]), (('Olga', 'Moscow'), [('Olga', 'Moscow')]), (('Petr', 'Samara'), [('Petr', 'Samara')]), (('Anna', 'Kazan'), [('Anna', 'Kazan')]), (('Dmitry', 'Samara'), [('Dmitry', 'Samara'), ('Dmitry', 'Samara')])]
RESULT: [(('Ivan', 'Samara'), ('Ivan', 'Samara')), (('Olga', 'Moscow'), ('Olga', 'Moscow')), (('Petr', 'Samara'), ('Petr', 'Samara')), (('Anna', 'Kazan'), ('Anna', 'Kazan')), (('Dmitry', 'Samara'), ('Dmitry', 'Samara'))]
PROJECTED: [('Ivan', 'Samara'), ('Olga', 'Moscow'), ('Petr', 'Samara'), ('Anna', 'Kazan'), ('Dmitry', 'Samara')]


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [37]:
from typing import Iterator

def RECORDREADER_union():
    for t in R:
        yield (t, t)
    for t in S:
        yield (t, t)

def MAP(t, _):
    yield (t, t)

def REDUCE(key, values: Iterator[tuple]):
    yield (key, key)

def flatten(nested):
    for it in nested:
        for x in it:
            yield x

def groupbykey(pairs):
    d = {}
    for k, v in pairs:
        d.setdefault(k, []).append(v)
    return d.items()

map_output = list(flatten(MAP(*x) for x in RECORDREADER_union()))
shuffle_output = list(groupbykey(map_output))
reduce_output = list(flatten(REDUCE(k, iter(vs)) for k, vs in shuffle_output))

print("MAP output:", map_output)
print("SHUFFLE output:", shuffle_output)
print("RESULT:", reduce_output)

union_tuples = [t for (t, _) in reduce_output]
print("UNION:", union_tuples)

MAP output: [(('Ivan', 25, 'Samara'), ('Ivan', 25, 'Samara')), (('Olga', 31, 'Moscow'), ('Olga', 31, 'Moscow')), (('Petr', 40, 'Samara'), ('Petr', 40, 'Samara')), (('Anna', 19, 'Kazan'), ('Anna', 19, 'Kazan')), (('Dmitry', 31, 'Samara'), ('Dmitry', 31, 'Samara')), (('Dmitry', 45, 'Samara'), ('Dmitry', 45, 'Samara')), (0, 0), (2, 2)]
SHUFFLE output: [(('Ivan', 25, 'Samara'), [('Ivan', 25, 'Samara')]), (('Olga', 31, 'Moscow'), [('Olga', 31, 'Moscow')]), (('Petr', 40, 'Samara'), [('Petr', 40, 'Samara')]), (('Anna', 19, 'Kazan'), [('Anna', 19, 'Kazan')]), (('Dmitry', 31, 'Samara'), [('Dmitry', 31, 'Samara')]), (('Dmitry', 45, 'Samara'), [('Dmitry', 45, 'Samara')]), (0, [0]), (2, [2])]
RESULT: [(('Ivan', 25, 'Samara'), ('Ivan', 25, 'Samara')), (('Olga', 31, 'Moscow'), ('Olga', 31, 'Moscow')), (('Petr', 40, 'Samara'), ('Petr', 40, 'Samara')), (('Anna', 19, 'Kazan'), ('Anna', 19, 'Kazan')), (('Dmitry', 31, 'Samara'), ('Dmitry', 31, 'Samara')), (('Dmitry', 45, 'Samara'), ('Dmitry', 45, 'Samara

### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [38]:
from typing import Iterator

R = [
    ("Ivan", 25, "Samara"),
    ("Olga", 31, "Moscow"),
    ("Petr", 40, "Samara"),
]

S = [
    ("Olga", 31, "Moscow"),
    ("Anna", 19, "Kazan"),
    ("Petr", 40, "Samara"),
]

def RECORDREADER_intersection():
    for t in R:
        yield (t, t)
    for t in S:
        yield (t, t)

def MAP(t, _):
    yield (t, t)

def REDUCE(key, values: Iterator[tuple]):
    vals = list(values)
    if len(vals) == 2:
        yield (key, key)
def flatten(nested):
    for it in nested:
        for x in it:
            yield x

def groupbykey(pairs):
    d = {}
    for k, v in pairs:
        d.setdefault(k, []).append(v)
    return d.items()

map_output = list(flatten(MAP(*x) for x in RECORDREADER_intersection()))
shuffle_output = list(groupbykey(map_output))
reduce_output = list(flatten(REDUCE(k, iter(vs)) for k, vs in shuffle_output))

print("MAP output:", map_output)
print("SHUFFLE output:", shuffle_output)
print("RESULT:", reduce_output)

intersection_tuples = [t for (t, _) in reduce_output]
print("INTERSECTION:", intersection_tuples)

MAP output: [(('Ivan', 25, 'Samara'), ('Ivan', 25, 'Samara')), (('Olga', 31, 'Moscow'), ('Olga', 31, 'Moscow')), (('Petr', 40, 'Samara'), ('Petr', 40, 'Samara')), (('Olga', 31, 'Moscow'), ('Olga', 31, 'Moscow')), (('Anna', 19, 'Kazan'), ('Anna', 19, 'Kazan')), (('Petr', 40, 'Samara'), ('Petr', 40, 'Samara'))]
SHUFFLE output: [(('Ivan', 25, 'Samara'), [('Ivan', 25, 'Samara')]), (('Olga', 31, 'Moscow'), [('Olga', 31, 'Moscow'), ('Olga', 31, 'Moscow')]), (('Petr', 40, 'Samara'), [('Petr', 40, 'Samara'), ('Petr', 40, 'Samara')]), (('Anna', 19, 'Kazan'), [('Anna', 19, 'Kazan')])]
RESULT: [(('Olga', 31, 'Moscow'), ('Olga', 31, 'Moscow')), (('Petr', 40, 'Samara'), ('Petr', 40, 'Samara'))]
INTERSECTION: [('Olga', 31, 'Moscow'), ('Petr', 40, 'Samara')]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [39]:
from typing import Iterator

R = [
    ("Ivan", 25, "Samara"),
    ("Olga", 31, "Moscow"),
    ("Petr", 40, "Samara"),
]

S = [
    ("Olga", 31, "Moscow"),
    ("Anna", 19, "Kazan"),
]

def RECORDREADER_difference():
    for t in R:
        yield (t, "R")
    for t in S:
        yield (t, "S")

def MAP(t, src):
    yield (t, src)

def REDUCE(key, values: Iterator[str]):
    tags = list(values)
    if tags == ["R"]:
        yield (key, key)
def flatten(nested):
    for it in nested:
        for x in it:
            yield x

def groupbykey(pairs):
    d = {}
    for k, v in pairs:
        d.setdefault(k, []).append(v)
    return d.items()

map_output = list(flatten(MAP(*x) for x in RECORDREADER_difference()))
shuffle_output = list(groupbykey(map_output))
reduce_output = list(flatten(REDUCE(k, iter(vs)) for k, vs in shuffle_output))

print("MAP output:", map_output)
print("SHUFFLE output:", shuffle_output)
print("RESULT:", reduce_output)

difference_tuples = [t for (t, _) in reduce_output]
print("DIFFERENCE (R \\ S):", difference_tuples)

MAP output: [(('Ivan', 25, 'Samara'), 'R'), (('Olga', 31, 'Moscow'), 'R'), (('Petr', 40, 'Samara'), 'R'), (('Olga', 31, 'Moscow'), 'S'), (('Anna', 19, 'Kazan'), 'S')]
SHUFFLE output: [(('Ivan', 25, 'Samara'), ['R']), (('Olga', 31, 'Moscow'), ['R', 'S']), (('Petr', 40, 'Samara'), ['R']), (('Anna', 19, 'Kazan'), ['S'])]
RESULT: [(('Ivan', 25, 'Samara'), ('Ivan', 25, 'Samara')), (('Petr', 40, 'Samara'), ('Petr', 40, 'Samara'))]
DIFFERENCE (R \ S): [('Ivan', 25, 'Samara'), ('Petr', 40, 'Samara')]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [40]:
from typing import Iterator

R = [
    (1, "x"),
    (2, "y"),
    (3, "x"),
]

S = [
    ("x", 100),
    ("x", 200),
    ("z", 300),
]

def RECORDREADER_join():
    for a, b in R:
        yield ("R", (a, b))
    for b, c in S:
        yield ("S", (b, c))


def MAP(src, t):
    if src == "R":
        a, b = t
        yield (b, ("R", a))
    else:
        b, c = t
        yield (b, ("S", c))

def REDUCE(b, values: Iterator[tuple]):
    a_list = []
    c_list = []

    for tag, x in values:
        if tag == "R":
            a_list.append(x)
        else:
            c_list.append(x)

    for a in a_list:
        for c in c_list:
            yield (a, b, c)

def flatten(nested):
    for it in nested:
        for x in it:
            yield x

def groupbykey(pairs):
    d = {}
    for k, v in pairs:
        d.setdefault(k, []).append(v)
    return d.items()

map_output = list(flatten(MAP(*x) for x in RECORDREADER_join()))
shuffle_output = list(groupbykey(map_output))
join_result = list(flatten(REDUCE(b, iter(vs)) for b, vs in shuffle_output))

print("MAP output:", map_output)
print("SHUFFLE output:", shuffle_output)
print("JOIN result (a,b,c):", join_result)

MAP output: [('x', ('R', 1)), ('y', ('R', 2)), ('x', ('R', 3)), ('x', ('S', 100)), ('x', ('S', 200)), ('z', ('S', 300))]
SHUFFLE output: [('x', [('R', 1), ('R', 3), ('S', 100), ('S', 200)]), ('y', [('R', 2)]), ('z', [('S', 300)])]
JOIN result (a,b,c): [(1, 'x', 100), (1, 'x', 200), (3, 'x', 100), (3, 'x', 200)]


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [41]:
from typing import Iterator
R = [
    ("Samara", 10, "x"),
    ("Moscow",  7, "y"),
    ("Samara",  3, "z"),
    ("Kazan",   5, "q"),
    ("Samara",  8, "w"),
    ("Moscow",  2, "p"),
]
def RECORDREADER():
    for t in R:
        yield (t, t)

def MAP(_, t):
    a, b, c = t
    yield (a, b)

def REDUCE(a, values: Iterator[int]):
    total = 0
    for v in values:
        total += v
    yield (a, total)
def flatten(nested):
    for it in nested:
        for x in it:
            yield x

def groupbykey(pairs):
    d = {}
    for k, v in pairs:
        d.setdefault(k, []).append(v)
    return d.items()


map_output = list(flatten(MAP(*x) for x in RECORDREADER()))
shuffle_output = list(groupbykey(map_output))
reduce_output = list(flatten(REDUCE(k, iter(vs)) for k, vs in shuffle_output))

print("MAP output:", map_output)
print("SHUFFLE output:", shuffle_output)
print("RESULT (SUM):", reduce_output)

MAP output: [('Samara', 10), ('Moscow', 7), ('Samara', 3), ('Kazan', 5), ('Samara', 8), ('Moscow', 2)]
SHUFFLE output: [('Samara', [10, 3, 8]), ('Moscow', [7, 2]), ('Kazan', [5])]
RESULT (SUM): [('Samara', 21), ('Moscow', 9), ('Kazan', 5)]


#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [42]:
from typing import Iterator

A = [
    (0, 0, 2), (0, 1, 1), (0, 2, 5),
    (1, 0, 7), (1, 1, 3), (1, 2, 4),
]

x = [
    (0, 10),
    (1, 20),
    (2, 30),
]

def flatten(nested):
    for it in nested:
        for e in it:
            yield e

def groupbykey(pairs):
    d = {}
    for k, v in pairs:
        d.setdefault(k, []).append(v)
    return d.items()

def RECORDREADER_job1():
    for i, j, a in A:
        yield ("A", (i, j, a))
    for j, xj in x:
        yield ("x", (j, xj))

def MAP1(src, rec):
    if src == "A":
        i, j, a = rec
        yield (j, ("A", i, a))
    else:
        j, xj = rec
        yield (j, ("x", xj))

def REDUCE1(j, values: Iterator[tuple]):
    xj = None
    a_entries = []

    for tag, *rest in values:
        if tag == "x":
            xj = rest[0]
        else:
            i, a = rest
            a_entries.append((i, a))

    if xj is None:
        return

    for i, a in a_entries:
        yield (i, a * xj)

map1_out = list(flatten(MAP1(*x) for x in RECORDREADER_job1()))
shuffle1 = list(groupbykey(map1_out))
job1_out = list(flatten(REDUCE1(j, iter(vs)) for j, vs in shuffle1))

print("JOB1 MAP output:", map1_out)
print("JOB1 SHUFFLE output:", shuffle1)
print("JOB1 RESULT (partials i, a*x):", job1_out)


def MAP2(i, partial):
    yield (i, partial)

def REDUCE2(i, partials: Iterator[int]):
    total = 0
    for p in partials:
        total += p
    yield (i, total)

map2_out = list(flatten(MAP2(*p) for p in job1_out))
shuffle2 = list(groupbykey(map2_out))
y = list(flatten(REDUCE2(i, iter(vs)) for i, vs in shuffle2))

print("JOB2 MAP output:", map2_out)
print("JOB2 SHUFFLE output:", shuffle2)
print("FINAL y:", y)

JOB1 MAP output: [(0, ('A', 0, 2)), (1, ('A', 0, 1)), (2, ('A', 0, 5)), (0, ('A', 1, 7)), (1, ('A', 1, 3)), (2, ('A', 1, 4)), (0, ('x', 10)), (1, ('x', 20)), (2, ('x', 30))]
JOB1 SHUFFLE output: [(0, [('A', 0, 2), ('A', 1, 7), ('x', 10)]), (1, [('A', 0, 1), ('A', 1, 3), ('x', 20)]), (2, [('A', 0, 5), ('A', 1, 4), ('x', 30)])]
JOB1 RESULT (partials i, a*x): [(0, 20), (1, 70), (0, 20), (1, 60), (0, 150), (1, 120)]
JOB2 MAP output: [(0, 20), (1, 70), (0, 20), (1, 60), (0, 150), (1, 120)]
JOB2 SHUFFLE output: [(0, [20, 20, 150]), (1, [70, 60, 120])]
FINAL y: [(0, 190), (1, 250)]


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [43]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [44]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J)
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
    (j, k) = k1
    b_jk = v1

    for i in range(small_mat.shape[0]):
        yield ((i, k), small_mat[i, j] * b_jk)

def REDUCE(key, values):
    s = 0.0
    for v in values:
        s += v
    yield (key, s)

Проверьте своё решение

In [45]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution))

True

In [46]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [47]:
import numpy as np

I, J, K = 3, 4, 5

np.random.seed(42)

A = np.random.randn(I, J)
B = np.random.randn(J, K)

def RECORDREADER_1():

    for i in range(I):
        for j in range(J):
            yield (("A", i, j), A[i, j])

    for j in range(J):
        for k in range(K):
            yield (("B", j, k), B[j, k])

def MAP_1(k1, v1):
    tag = k1[0]
    if tag == "A":
        _, i, j = k1
        a_ij = v1
        yield (j, ("A", i, a_ij))
    else:
        _, j, k = k1
        b_jk = v1
        yield (j, ("B", k, b_jk))

def REDUCE_1(j, values):
    A_list = []
    B_list = []

    for tag, idx, val in values:
        if tag == "A":
            A_list.append((idx, val))
        else:
            B_list.append((idx, val))

    for i, a_ij in A_list:
        for k, b_jk in B_list:
            yield ((i, k), a_ij * b_jk)

partials = list(MapReduce(RECORDREADER_1, MAP_1, REDUCE_1))

def RECORDREADER_2():
    for key, val in partials:
        yield (key, val)

def MAP_2(key, val):
    yield (key, val)

def REDUCE_2(key, values):
    s = 0.0
    for v in values:
        s += v
    yield (key, s)

C_items = list(MapReduce(RECORDREADER_2, MAP_2, REDUCE_2))

C_mr = np.zeros((I, K), dtype=float)
for (i, k), v in C_items:
    C_mr[i, k] = v

C_np = A @ B

matches = np.allclose(C_mr, C_np, rtol=1e-9, atol=1e-9)
print("Matches numpy:", matches)

np.set_printoptions(precision=8, suppress=False)
print(repr(C_mr))


Matches numpy: True
array([[ 0.69267344, -2.66238979, -1.45836866, -1.32651692,  1.60371888],
       [ 0.26473192, -2.05032856, -0.34898427, -0.4981012 , -0.1061408 ],
       [-0.14936455,  1.34556731,  0.43167223,  1.28801123,  0.02373079]])


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [50]:
from typing import Iterator, Tuple, Iterable, List
import numpy as np

np.random.seed(42)

I, J, K = 3, 4, 6
A = np.random.randn(I, J).astype(float)
B = np.random.randn(J, K).astype(float)

reference = A @ B

n_maps = 4

def chunkify(items: List, n_chunks: int) -> List[List]:
    n_chunks = max(1, int(n_chunks))
    out = [[] for _ in range(n_chunks)]
    for idx, x in enumerate(items):
        out[idx % n_chunks].append(x)
    return out

def RECORDREADER_A() -> Iterable[Tuple[Tuple[str, int, int], float]]:
    for i in range(I):
        for j in range(J):
            yield (("A", i, j), float(A[i, j]))

def RECORDREADER_B() -> Iterable[Tuple[Tuple[str, int, int], float]]:
    for j in range(J):
        for k in range(K):
            yield (("B", j, k), float(B[j, k]))

def INPUTFORMAT1():
    a_items = list(RECORDREADER_A())
    b_items = list(RECORDREADER_B())

    a_maps = max(1, n_maps // 2)
    b_maps = max(1, n_maps - a_maps)

    a_chunks = chunkify(a_items, a_maps)
    b_chunks = chunkify(b_items, b_maps)

    def rr_from_chunk(chunk):
        for k1, v1 in chunk:
            yield (k1, v1)


    for ch in a_chunks:
        yield rr_from_chunk(ch)
    for ch in b_chunks:
        yield rr_from_chunk(ch)

def MAP1(k1, v1):
    tag = k1[0]
    if tag == "A":
        _, i, j = k1
        yield (j, ("A", i, float(v1)))
    else:
        _, j, k = k1
        yield (j, ("B", k, float(v1)))

def REDUCE1(j: int, vals: Iterator[tuple]):
    left = []
    right = []

    for rec in vals:
        if rec[0] == "A":
            _, i, aij = rec
            left.append((i, aij))
        else:
            _, k, bjk = rec
            right.append((k, bjk))

    for i, aij in left:
        for k, bjk in right:
            yield ((i, k), aij * bjk)

stage1 = MapReduceDistributed(INPUTFORMAT1, MAP1, REDUCE1)
stage1 = [(pid, list(part)) for (pid, part) in stage1]
partials = [kv for (_, part) in stage1 for kv in part]

print("partials:", len(partials), "expected:", I * J * K)

def INPUTFORMAT2():
    chunks = chunkify(partials, n_maps)

    def rr_from_chunk(chunk):
        for (ik, val) in chunk:
            yield (ik, float(val))

    for ch in chunks:
        yield rr_from_chunk(ch)

def MAP2(ik, val: float):
    yield (ik, float(val))

def REDUCE2(ik, vals: Iterator[float]):
    s = 0.0
    for x in vals:
        s += x
    yield (ik, s)

stage2 = MapReduceDistributed(INPUTFORMAT2, MAP2, REDUCE2, COMBINER=REDUCE2)
stage2 = [(pid, list(part)) for (pid, part) in stage2]
out = [kv for (_, part) in stage2 for kv in part]

def to_dense(pairs, I: int, K: int):
    mat = np.zeros((I, K), dtype=float)
    for (i, k), v in pairs:
        mat[i, k] = v
    return mat

solution = to_dense(out, I, K)

print("Matches numpy:", np.allclose(reference, solution))
np.set_printoptions(precision=8, suppress=False)
print(repr(solution))
print(repr(reference))


36 key-value pairs were sent over a network.
partials: 72 expected: 72
36 key-value pairs were sent over a network.
Matches numpy: True
array([[-1.02327419,  2.13783608, -1.82548003, -1.6156694 ,  0.35130937,
        -1.69522512],
       [-1.16552101,  2.3753465 , -1.76728926, -0.03389288, -0.09593981,
        -1.13756019],
       [-0.07374341, -0.78208804,  2.14468405,  0.45998473,  0.40739933,
        -0.21678325]])
array([[-1.02327419,  2.13783608, -1.82548003, -1.6156694 ,  0.35130937,
        -1.69522512],
       [-1.16552101,  2.3753465 , -1.76728926, -0.03389288, -0.09593981,
        -1.13756019],
       [-0.07374341, -0.78208804,  2.14468405,  0.45998473,  0.40739933,
        -0.21678325]])


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [51]:
import numpy as np
import math
from typing import Iterator, List, Tuple

np.random.seed(4)

I, J, K = 4, 5, 3
A = np.random.randn(I, J).astype(float)
B = np.random.randn(J, K).astype(float)
reference = A @ B

maps = 6
reducers = 3

A_entries = [(("A", i, j), float(A[i, j])) for i in range(I) for j in range(J)]
B_entries = [(("B", j, k), float(B[j, k])) for j in range(J) for k in range(K)]

def random_partition_full(entries: List[Tuple[tuple, float]], n_parts: int, seed: int):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(entries))
    rng.shuffle(idx)
    parts = np.array_split(idx, n_parts)
    return [[entries[i] for i in part] for part in parts]

A_splits = random_partition_full(A_entries, n_parts=3, seed=10)
B_splits = random_partition_full(B_entries, n_parts=3, seed=20)

def INPUTFORMAT1():
    def RR(split):
        for kv in split:
            yield kv
    for sp in A_splits:
        yield RR(sp)
    for sp in B_splits:
        yield RR(sp)

def MAP1(k1, v1):
    tag = k1[0]
    if tag == "A":
        _, i, j = k1
        yield (j, ("A", i, float(v1)))
    else:
        _, j, k = k1
        yield (j, ("B", k, float(v1)))

def REDUCE1(j: int, tagged_vals: Iterator[tuple]):
    left = []
    right = []
    for rec in tagged_vals:
        if rec[0] == "A":
            _, i, aij = rec
            left.append((i, aij))
        else:
            _, k, bjk = rec
            right.append((k, bjk))

    for i, aij in left:
        for k, bjk in right:
            yield ((i, k), aij * bjk)

stage1 = MapReduceDistributed(INPUTFORMAT1, MAP1, REDUCE1, COMBINER=None)
stage1 = [(pid, list(part)) for (pid, part) in stage1]
partials = [kv for (_, part) in stage1 for kv in part]

print("CASE1 partials:", len(partials), "expected:", I * J * K)

maps = 4
reducers = 3

def split_random(items, n_parts, seed):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(items))
    rng.shuffle(idx)
    parts = np.array_split(idx, n_parts)

    def RR(index_chunk):
        for t in index_chunk:
            yield items[t]
    for part in parts:
        yield RR(part)

def INPUTFORMAT2():
    return split_random(partials, n_parts=maps, seed=30)

def MAP2(key, val):
    yield (key, float(val))

def REDUCE2(key, vals: Iterator[float]):
    s = 0.0
    for x in vals:
        s += x
    yield (key, s)

stage2 = MapReduceDistributed(INPUTFORMAT2, MAP2, REDUCE2, COMBINER=REDUCE2)
stage2 = [(pid, list(part)) for (pid, part) in stage2]
out = [kv for (_, part) in stage2 for kv in part]

def asmatrix(pairs, I, K):
    mat = np.zeros((I, K), dtype=float)
    for ((i, k), v) in pairs:
        mat[i, k] = v
    return mat

solution = asmatrix(out, I, K)

print("Matches numpy (full cover, random RR):", np.allclose(reference, solution))
np.set_printoptions(precision=8, suppress=False)
print(repr(solution))
print(repr(reference))

maps = 6
reducers = 3

def random_subset(entries, frac, seed):
    rng = np.random.default_rng(seed)
    n = len(entries)
    m = int(np.floor(frac * n))
    idx = rng.choice(n, size=m, replace=False)
    return [entries[i] for i in idx]

A_subset = random_subset(A_entries, frac=0.7, seed=111)
B_subset = random_subset(B_entries, frac=0.7, seed=222)

A_splits2 = random_partition_full(A_subset, n_parts=3, seed=333)
B_splits2 = random_partition_full(B_subset, n_parts=3, seed=444)


def INPUTFORMAT1_bad():
    def RR(split):
        for kv in split:
            yield kv
    for sp in A_splits2:
        yield RR(sp)
    for sp in B_splits2:
        yield RR(sp)

stage1_bad = MapReduceDistributed(INPUTFORMAT1_bad, MAP1, REDUCE1, COMBINER=None)
stage1_bad = [(pid, list(part)) for (pid, part) in stage1_bad]
partials_bad = [kv for (_, part) in stage1_bad for kv in part]
print("\nCASE2 partials (subset):", len(partials_bad), "expected(full):", I * J * K)

maps = 4
reducers = 3

def INPUTFORMAT2_bad():
    return split_random(partials_bad, n_parts=maps, seed=555)

stage2_bad = MapReduceDistributed(INPUTFORMAT2_bad, MAP2, REDUCE2, COMBINER=REDUCE2)
stage2_bad = [(pid, list(part)) for (pid, part) in stage2_bad]
out_bad = [kv for (_, part) in stage2_bad for kv in part]
solution_bad = asmatrix(out_bad, I, K)

print("Matches numpy (random SUBSET, losses):", np.allclose(reference, solution_bad))
print("Max abs diff:", float(np.max(np.abs(reference - solution_bad))))


35 key-value pairs were sent over a network.
CASE1 partials: 60 expected: 60
36 key-value pairs were sent over a network.
Matches numpy (full cover, random RR): True
array([[-3.30359593,  2.7572802 , -0.5851318 ],
       [ 2.8464065 ,  0.29702452,  1.91922881],
       [-0.29012046,  1.87650961,  2.21872797],
       [ 2.02448215, -3.63157051,  2.6886677 ]])
array([[-3.30359593,  2.7572802 , -0.5851318 ],
       [ 2.8464065 ,  0.29702452,  1.91922881],
       [-0.29012046,  1.87650961,  2.21872797],
       [ 2.02448215, -3.63157051,  2.6886677 ]])
24 key-value pairs were sent over a network.

CASE2 partials (subset): 26 expected(full): 60
24 key-value pairs were sent over a network.
Matches numpy (random SUBSET, losses): False
Max abs diff: 2.5659071726590548
